# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their IDs, along with fields
print("Available record sets:\n")
for record_set in dataset.record_sets:
    print(f"RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {record_set.description}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis using the record set and field `@id`s from the overview.

In [ ]:
# --- Configure the record set ids based on previous cell output ---
# For demonstration, we'll fetch all record sets discovered above.

record_set_ids = [record_set.id for record_set in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # load records for each record set
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet '@id': {rs_id}")
    else:
        print(f"No records found for RecordSet '@id': {rs_id}")

# For analysis, choose one record set (here we simply pick the first available one)
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nAvailable columns for RecordSet '@id': {first_rs_id}")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes were loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose the record set and field IDs to work with
record_set_id = None
numeric_field_id = None
group_field_id = None

# For demonstration, automatically select a numeric field if possible
import numpy as np

for rs_id, df in dataframes.items():
    # Try to find first numeric field
    for col in df.columns:
        # Consider only float or int columns
        if np.issubdtype(df[col].dtype, np.number):
            record_set_id = rs_id
            numeric_field_id = col
            # Try to find a non-numeric field as group
            for gcol in df.columns:
                if gcol != col and not np.issubdtype(df[gcol].dtype, np.number):
                    group_field_id = gcol
                    break
            break
    if record_set_id and numeric_field_id:
        break

if not record_set_id:
    print("No numeric fields found in available dataframes for EDA.")
else:
    print(f"Using RecordSet '@id': {record_set_id}")
    print(f"Numeric field for analysis: {numeric_field_id}")
    if group_field_id:
        print(f"Grouping field: {group_field_id}")

    df = dataframes[record_set_id]
    # Filtering out records where the numeric field is above threshold
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10  # Use mean as a dynamic threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution and grouped means if available
if record_set_id and numeric_field_id:
    df = dataframes[record_set_id].copy()
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Key takeaways:**

- The dataset provides mass spectrometry protein profiling results from extracellular vesicles isolated from human mast cells, with rich metadata and detailed record set structure.
- Fields and columns are clearly discoverable via Croissant `@id`, ensuring reliable, reproducible access and EDA pipelines.
- Further analysis can be performed on individual record sets using the IDs listed above, and this notebook can be extended to cover domain-specific tasks.
- The workflow here supports FAIR data principles, and leverages the Croissant schema for robust scientific data handling.